In [19]:
"""
Rat Pose Annotator with YOLO Training Integration

Requirements:
- Python 3.8-3.10 (recommended)
- OpenCV: pip install opencv-python
- NumPy < 2.0: pip install 'numpy<2'
- Pillow: pip install pillow
- PyYAML: pip install pyyaml
- Ultralytics: pip install ultralytics
- Matplotlib: pip install matplotlib

If you get NumPy compatibility errors, run:
pip uninstall numpy
pip install 'numpy<2'
pip install ultralytics --upgrade

For GPU support (optional):
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
"""

"\nRat Pose Annotator with YOLO Training Integration\n\nRequirements:\n- Python 3.8-3.10 (recommended)\n- OpenCV: pip install opencv-python\n- NumPy < 2.0: pip install 'numpy<2'\n- Pillow: pip install pillow\n- PyYAML: pip install pyyaml\n- Ultralytics: pip install ultralytics\n- Matplotlib: pip install matplotlib\n\nIf you get NumPy compatibility errors, run:\npip uninstall numpy\npip install 'numpy<2'\npip install ultralytics --upgrade\n\nFor GPU support (optional):\npip install torch torchvision --index-url https://download.pytorch.org/whl/cu118\n"

In [20]:
import cv2
import numpy as np
import json
import os
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk
import threading
from datetime import datetime
import subprocess
import tempfile
import shutil
import yaml
import sys
import queue
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib
matplotlib.use('TkAgg')

from tips import get_all_tips

In [21]:
# Check NumPy version
if hasattr(np, '__version__'):
    np_version = tuple(map(int, np.__version__.split('.')[:2]))
    if np_version >= (2, 0):
        print(f"Warning: NumPy {np.__version__} detected. Some packages may require NumPy < 2.0")
        print("If you encounter errors, run: pip install 'numpy<2'")

# Try to import YOLO
try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError as e:
    YOLO_AVAILABLE = False
    print("\n" + "="*60)
    print("YOLO Installation Required")
    print("="*60)
    print("ultralytics is not installed or there's a dependency issue.")
    print("\nTo fix, run these commands in order:")
    print("1. pip uninstall numpy ultralytics")
    print("2. pip install 'numpy<2'")
    print("3. pip install ultralytics")
    print("\nTraining features will be disabled until fixed.")
    print("="*60 + "\n")

If you encounter errors, run: pip install 'numpy<2'


In [22]:
### Sub-routines

In [23]:

class RatPoseAnnotator:
    def __init__(self, root):
        self.root = root
        self.root.title("Rat Pose Annotator")
        self.root.geometry("1200x800")
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)  
        self.keypoint_names = ['head', 'center', 'tail', 'right_front_paw', 
                        'left_front_paw', 'right_back_paw', 'left_back_paw']
        
        # Video variables
        self.cap = None
        self.current_frame = 0
        self.total_frames = 0
        self.fps = 30
        self.playing = False
        self.video_filename = None
        
        # Annotation variables
        self.annotations = {}
        self.current_points = {'head': None, 'center': None, 'tail': None}
        self.selected_point = 'head'
        self.colors = {
            'head': (255, 0, 0),      
            'center': (0, 255, 0),    
            'tail': (0, 0, 255),
            'right_front_paw': (255, 255, 0),
            'left_front_paw': (255, 0, 255),
            'right_back_paw': (0, 255, 255),
            'left_back_paw': (128, 128, 128)
        }
        self.point_radius = 8
        self.show_skeleton = True
        self.copy_previous = True
        
        # Drag variables
        self.dragging = False
        self.drag_point = None
        self.scale_x = 1.0
        self.scale_y = 1.0
        
        # Training variables
        self.model_path = None
        self.yolo_model = None  # Add YOLO model instance
        self._last_saved_annotations = {}
        self.confidence_threshold = 0.5
        self.auto_advance_low_conf = True
        self.model_iteration = 0
        
        # Training history for mAP tracking
        self.training_history = []
        
        # Prediction tracking
        self.predictions = {}
        self.prediction_confidence = {}
        
        # Click mode variable
        self.click_mode = False
        
        # Thread communication
        self.training_queue = queue.Queue()
        self.is_training = False
        
        self.setup_gui()
        self.check_training_status()

    def on_closing(self):
        """Handle window closing event"""
        if self.annotations and hasattr(self, '_last_saved_annotations'):
            # Check if annotations have changed since last save
            if self.annotations != self._last_saved_annotations:
                result = messagebox.askyesnocancel(
                    "Unsaved Changes",
                    "Warning: You have unsaved annotations.\n\nDo you want to save before exiting?"
                )
                if result is True:  # Yes - save and exit
                    self.save_annotations()
                    self.root.destroy()
                elif result is False:  # No - exit without saving
                    self.root.destroy()
                # else: Cancel - do nothing
            else:
                self.root.destroy()
        elif self.annotations and not hasattr(self, '_last_saved_annotations'):
            # Has annotations but never saved
            result = messagebox.askyesnocancel(
                "Unsaved Changes", 
                "Warning: You have unsaved annotations.\n\nDo you want to save before exiting?"
            )
            if result is True:  # Yes - save and exit
                self.save_annotations()
                self.root.destroy()
            elif result is False:  # No - exit without saving
                self.root.destroy()
            # else: Cancel - do nothing
        else:
            # No annotations, just close
            self.root.destroy()
        
    def check_training_status(self):
        """Check for training updates from background thread"""
        try:
            while True:
                message = self.training_queue.get_nowait()
                if message['type'] == 'status':
                    self.model_status.set(message['text'])
                elif message['type'] == 'complete':
                    self.is_training = False
                    self.yolo_model = message['model']
                    self.model_path = message['path']
                    self.model_status.set(f"Model ready: {message['name']}")
                    
                    # Add to training history
                    self.training_history.append({
                        'iteration': self.model_iteration,
                        'frames': message['frames'],
                        'mAP': message.get('mAP', 0),
                        'timestamp': datetime.now()
                    })
                    
                    messagebox.showinfo("Success", message['message'])
                elif message['type'] == 'error':
                    self.is_training = False
                    self.model_status.set("Training failed")
                    messagebox.showerror("Training Error", message['message'])
        except queue.Empty:
            pass
        
        # Schedule next check
        self.root.after(100, self.check_training_status)

    def auto_pair_from_folder(self):
        """Auto-pair annotation and video files from a single folder"""
        # Ask user to select the folder containing both .json and .avi files
        source_folder = filedialog.askdirectory(
            title="Select folder containing paired .json and .avi files"
        )
        if not source_folder:
            return
        
        # Find all json and video files
        json_files = {}
        video_files = {}
        
        for file in os.listdir(source_folder):
            if file.endswith('.json'):
                base_name = os.path.splitext(file)[0]
                json_files[base_name] = os.path.join(source_folder, file)
            elif file.endswith(('.avi', '.mp4', '.mov')):
                base_name = os.path.splitext(file)[0]
                video_files[base_name] = os.path.join(source_folder, file)
        
        # Find matches
        matched_pairs = []
        unmatched_json = []
        unmatched_video = []
        
        for base_name, json_path in json_files.items():
            if base_name in video_files:
                matched_pairs.append({
                    'annotation': json_path,
                    'video': video_files[base_name],
                    'base_name': base_name
                })
            else:
                unmatched_json.append(base_name)
        
        for base_name in video_files:
            if base_name not in json_files:
                unmatched_video.append(base_name)
        
        # Report findings
        message = f"Found {len(matched_pairs)} matched pairs\n"
        if unmatched_json:
            message += f"\n{len(unmatched_json)} JSON files without matching videos:\n"
            message += "\n".join(f"  - {name}" for name in unmatched_json[:5])
            if len(unmatched_json) > 5:
                message += f"\n  ... and {len(unmatched_json) - 5} more"
        if unmatched_video:
            message += f"\n\n{len(unmatched_video)} videos without matching JSON:\n"
            message += "\n".join(f"  - {name}" for name in unmatched_video[:5])
            if len(unmatched_video) > 5:
                message += f"\n  ... and {len(unmatched_video) - 5} more"
        
        if not matched_pairs:
            messagebox.showwarning("No Matches", "No matching annotation-video pairs found!")
            return
        
        # Ask if user wants to proceed
        if not messagebox.askyesno("Confirm Auto-Pairing", 
                                message + "\n\nProceed with matched pairs?"):
            return
        
        # Now open the merge dialog with pre-populated pairs
        self.merge_annotations_with_pairs(matched_pairs)
        
    def setup_gui(self):
        # Menu bar
        menubar = tk.Menu(self.root)
        self.root.config(menu=menubar)
        
        file_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="File", menu=file_menu)
        file_menu.add_command(label="Open Video", command=self.open_video)
        file_menu.add_command(label="Save Annotations", command=self.save_annotations)
        file_menu.add_command(label="Load Annotations", command=self.load_annotations)
        file_menu.add_command(label="Merge Multiple Annotations", command=self.merge_annotations)
        file_menu.add_separator()
        file_menu.add_command(label="Export for Training", command=self.export_for_training)
        file_menu.add_separator()
        file_menu.add_command(label="Exit", command=self.on_closing)    

        tools_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="Tools", menu=tools_menu)
        tools_menu.add_command(label="Train from Folder", command=self.train_from_folder)
        tools_menu.add_command(label="Load Model", command=self.load_model)
        tools_menu.add_command(label="Predict Next Frames", command=self.predict_frames)
        tools_menu.add_command(label="Review Predictions", command=self.review_predictions)
        tools_menu.add_command(label="Show Model Stats", command=self.show_model_stats)
        tools_menu.add_command(label="Show Training Progress", command=self.show_training_progress)

        # Help menu
        help_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="Help", menu=help_menu)
        help_menu.add_command(label="Keyboard Shortcuts", command=lambda: self.show_help("Keyboard Shortcuts"))
        help_menu.add_command(label="Labeling Workflow", command=lambda: self.show_help("Labeling Workflow"))
        help_menu.add_command(label="Iterative Training", command=lambda: self.show_help("Iterative Training"))
        help_menu.add_command(label="Multiple Videos", command=lambda: self.show_help("Multiple Videos"))
        help_menu.add_command(label="Advanced Tips", command=lambda: self.show_help("Advanced Tips"))
        help_menu.add_separator()
        help_menu.add_command(label="Show All Tips", command=self.show_all_tips)
        
        # Main container
        main_frame = ttk.Frame(self.root)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Left panel - Video display
        video_frame = ttk.LabelFrame(main_frame, text="Video", padding=10)
        video_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        self.video_label = tk.Label(video_frame, bg='black')
        self.video_label.pack(fill=tk.BOTH, expand=True)
        self.video_label.bind("<Button-1>", self.on_video_click)
        self.video_label.bind("<B1-Motion>", self.on_drag)
        self.video_label.bind("<ButtonRelease-1>", self.on_release)
        
        # Video controls
        controls_frame = ttk.Frame(video_frame)
        controls_frame.pack(fill=tk.X, pady=(10, 0))
        
        self.play_button = ttk.Button(controls_frame, text="Play", command=self.toggle_play)
        self.play_button.pack(side=tk.LEFT, padx=5)
        
        ttk.Button(controls_frame, text="<<", command=lambda: self.skip_frames(-10)).pack(side=tk.LEFT)
        ttk.Button(controls_frame, text="<", command=lambda: self.skip_frames(-1)).pack(side=tk.LEFT)
        ttk.Button(controls_frame, text=">", command=lambda: self.skip_frames(1)).pack(side=tk.LEFT)
        ttk.Button(controls_frame, text=">>", command=lambda: self.skip_frames(10)).pack(side=tk.LEFT)
        
        # Frame slider
        self.frame_var = tk.IntVar()
        self.frame_slider = ttk.Scale(video_frame, from_=0, to=100, 
                                      orient=tk.HORIZONTAL, variable=self.frame_var,
                                      command=self.on_slider_change)
        self.frame_slider.pack(fill=tk.X, pady=(10, 0))
        
        # Frame info
        self.frame_info = tk.StringVar(value="No video loaded")
        ttk.Label(video_frame, textvariable=self.frame_info).pack()
        
        # Progress tracker for annotations
        progress_frame = ttk.Frame(video_frame)
        progress_frame.pack(fill=tk.X, pady=(10, 0))
        ttk.Label(progress_frame, text="Annotation Progress:").pack(anchor=tk.W)
        
        # Canvas for progress bar - ensure it's properly constrained
        progress_container = ttk.Frame(progress_frame)
        progress_container.pack(fill=tk.X, pady=(5, 0))
        
        self.progress_canvas = tk.Canvas(progress_container, height=20, bg='white', 
                                        highlightthickness=1, relief=tk.SUNKEN)
        self.progress_canvas.pack(fill=tk.X, expand=True)
        
        # Right panel - Controls with scrollbar
        control_container = ttk.Frame(main_frame)
        control_container.pack(side=tk.RIGHT, fill=tk.Y, padx=(10, 0))
        
        # Create canvas and scrollbar for scrollable content
        canvas = tk.Canvas(control_container, width=300)
        scrollbar = ttk.Scrollbar(control_container, orient="vertical", command=canvas.yview)
        scrollable_frame = ttk.Frame(canvas)
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Control frame now goes inside scrollable_frame
        control_frame = ttk.LabelFrame(scrollable_frame, text="Controls", padding=10)
        control_frame.pack(fill=tk.X, padx=5, pady=5)
        
        # Point selection
        point_frame = ttk.LabelFrame(control_frame, text="Select Point to Mark", padding=10)
        point_frame.pack(fill=tk.X, pady=(0, 10))
        
        self.point_var = tk.StringVar(value='head')
        self.radio_buttons = {}
        for i, point in enumerate(self.keypoint_names):
            rb_frame = ttk.Frame(point_frame)
            rb_frame.pack(fill=tk.X, pady=2)
            
            rb = ttk.Radiobutton(rb_frame, text=point.capitalize(), 
                                 variable=self.point_var, value=point,
                                 command=self.on_point_select)
            rb.pack(side=tk.LEFT)
            
            # Color indicator
            color = self.colors[point]
            color_hex = f'#{color[2]:02x}{color[1]:02x}{color[0]:02x}'
            color_label = tk.Label(rb_frame, text="●", fg=color_hex, font=('Arial', 16))
            color_label.pack(side=tk.LEFT, padx=(10, 0))
            
            self.radio_buttons[point] = rb
        
        # Current positions
        pos_frame = ttk.LabelFrame(control_frame, text="Current Positions", padding=10)
        pos_frame.pack(fill=tk.X, pady=(0, 10))
        
        self.pos_labels = {}
        for point in self.keypoint_names:
            frame = ttk.Frame(pos_frame)
            frame.pack(fill=tk.X, pady=2)
            ttk.Label(frame, text=f"{point.capitalize()}:", width=8).pack(side=tk.LEFT)
            self.pos_labels[point] = tk.StringVar(value="Not set")
            ttk.Label(frame, textvariable=self.pos_labels[point], width=15).pack(side=tk.LEFT)
        
        # Options
        options_frame = ttk.LabelFrame(control_frame, text="Options", padding=10)
        options_frame.pack(fill=tk.X, pady=(0, 10))
        
        self.skeleton_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(options_frame, text="Show skeleton", 
                        variable=self.skeleton_var,
                        command=self.update_frame).pack(anchor=tk.W)
        
        self.copy_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(options_frame, text="Copy from previous frame", 
                        variable=self.copy_var).pack(anchor=tk.W)
        
        self.click_mode_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(options_frame, text="Click to move mode (vs drag)", 
                        variable=self.click_mode_var,
                        command=self.toggle_click_mode).pack(anchor=tk.W)
        
        # Training/Model frame
        model_frame = ttk.LabelFrame(control_frame, text="Model Training", padding=10)
        model_frame.pack(fill=tk.X, pady=(0, 10))
        
        self.model_status = tk.StringVar(value="No model loaded")
        ttk.Label(model_frame, textvariable=self.model_status).pack()
        
        ttk.Button(model_frame, text="Quick Train...", 
                   command=self.quick_train_dialog,
                   state=tk.NORMAL if YOLO_AVAILABLE else tk.DISABLED).pack(fill=tk.X, pady=2)
        ttk.Button(model_frame, text="Predict Frames...", 
                   command=self.predict_frames_dialog).pack(fill=tk.X, pady=2)
        
        # Model loading button
        ttk.Button(model_frame, text="Load Trained Model", 
                   command=self.load_model).pack(fill=tk.X, pady=2)
        
        # Confidence threshold
        conf_frame = ttk.Frame(model_frame)
        conf_frame.pack(fill=tk.X, pady=(5, 0))
        ttk.Label(conf_frame, text="Confidence:").pack(side=tk.LEFT)
        self.conf_var = tk.DoubleVar(value=0.5)
        ttk.Scale(conf_frame, from_=0.1, to=0.9, variable=self.conf_var,
                 orient=tk.HORIZONTAL, length=150).pack(side=tk.LEFT, padx=5)
        self.conf_label = ttk.Label(conf_frame, text="0.50")
        self.conf_label.pack(side=tk.LEFT)
        self.conf_var.trace('w', lambda *args: self.conf_label.config(text=f"{self.conf_var.get():.2f}"))
        
        # Actions
        action_frame = ttk.LabelFrame(control_frame, text="Actions", padding=10)
        action_frame.pack(fill=tk.X, pady=(0, 10))
        
        ttk.Button(action_frame, text="Clear Current Frame", 
                   command=self.clear_current_frame).pack(fill=tk.X, pady=2)
        ttk.Button(action_frame, text="Mass Clear Frames...", 
                   command=self.mass_clear_dialog).pack(fill=tk.X, pady=2)
        ttk.Button(action_frame, text="Copy Previous to Current", 
                   command=self.copy_previous_frame).pack(fill=tk.X, pady=2)
        ttk.Button(action_frame, text="Interpolate Between Keyframes", 
                   command=self.interpolate_frames).pack(fill=tk.X, pady=2)
        ttk.Button(action_frame, text="Auto-track from Current", 
                   command=self.auto_track_forward).pack(fill=tk.X, pady=2)
        ttk.Button(action_frame, text="Jump to Frame...", 
                   command=self.jump_to_frame_dialog).pack(fill=tk.X, pady=2)
        
        # Statistics
        stats_frame = ttk.LabelFrame(control_frame, text="Statistics", padding=10)
        stats_frame.pack(fill=tk.X)
        
        self.stats_var = tk.StringVar(value="No annotations")
        ttk.Label(stats_frame, textvariable=self.stats_var, justify=tk.LEFT).pack()
        
        # Keyboard shortcuts info
        shortcuts_frame = ttk.LabelFrame(control_frame, text="Shortcuts", padding=10)
        shortcuts_frame.pack(fill=tk.X, pady=(10, 0))
        
        shortcuts = [
            "Click: Place point",
            "Drag: Move point",
            "Space: Play/Pause",
            "A/D: Previous/Next frame",
            "W/S: Skip ±35 frames",
            "1/2/3: Select Head/Center/Tail",
            "C: Copy previous frame",
            "R: Review mode (low conf frames)",
            "Delete: Clear frame",
            "I: Interpolate",
            "G: Go to frame (jump)"
        ]
        for shortcut in shortcuts:
            ttk.Label(shortcuts_frame, text=shortcut, font=('Arial', 9)).pack(anchor=tk.W)
        
        # Bind keyboard shortcuts
        self.root.bind('<space>', lambda e: self.toggle_play())
        self.root.bind('a', lambda e: self.skip_frames(-1))
        self.root.bind('d', lambda e: self.skip_frames(1))
        self.root.bind('w', lambda e: self.skip_frames(-35))
        self.root.bind('.', lambda e: self.skip_frames(35))

        # Dynamic keypoint bindings
        for i, keypoint in enumerate(self.keypoint_names):
            self.root.bind(str(i+1), lambda e, kp=keypoint: self.point_var.set(kp))
            
        self.root.bind('c', lambda e: self.copy_previous_frame())
        self.root.bind('r', lambda e: self.review_predictions())
        self.root.bind('<Delete>', lambda e: self.clear_current_frame())
        self.root.bind('i', lambda e: self.interpolate_frames())
        self.root.bind('g', lambda e: self.jump_to_frame_dialog())
    
    def open_video(self):
        filename = filedialog.askopenfilename(
            title="Select video file",
            filetypes=[("Video files", "*.avi *.mp4 *.mov"), ("All files", "*.*")]
        )
        if filename:
            self.load_video(filename)
    
    def load_video(self, filename):
        if self.cap:
            self.cap.release()
        
        self.video_filename = filename
        
        self.cap = cv2.VideoCapture(filename)
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.fps = int(self.cap.get(cv2.CAP_PROP_FPS))
        self.current_frame = 0
        
        self.frame_slider.config(to=self.total_frames-1)
        self.annotations = {}
        self.predictions = {}
        self.prediction_confidence = {}
        
        self.update_frame()
        self.update_progress_bar()
    
    def update_frame(self):
        if not self.cap:
            return
        
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, self.current_frame)
        ret, frame = self.cap.read()
        
        if ret:
            # Store original frame dimensions
            self.frame_height, self.frame_width = frame.shape[:2]
            
            # Copy previous frame's annotations if enabled and current frame has none
            if (self.copy_var.get() and 
                self.current_frame not in self.annotations and 
                self.current_frame > 0 and 
                self.current_frame - 1 in self.annotations):
                self.annotations[self.current_frame] = self.annotations[self.current_frame - 1].copy()
            
            # Draw annotations
            display_frame = self.draw_annotations(frame)
            
            # Convert to RGB and resize for display
            display_frame = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)
            
            # Resize to fit in window
            max_width = 800
            max_height = 600
            scale_x = min(max_width / self.frame_width, 1.0)
            scale_y = min(max_height / self.frame_height, 1.0)
            scale = min(scale_x, scale_y)
            
            new_width = int(self.frame_width * scale)
            new_height = int(self.frame_height * scale)
            display_frame = cv2.resize(display_frame, (new_width, new_height))
            
            # Store scale factors for coordinate conversion
            self.display_width = new_width
            self.display_height = new_height
            
            # Convert to PhotoImage
            image = Image.fromarray(display_frame)
            photo = ImageTk.PhotoImage(image=image)
            
            self.video_label.config(image=photo)
            self.video_label.image = photo
            
            # Update info
            self.frame_var.set(self.current_frame)
            self.frame_info.set(f"Frame {self.current_frame}/{self.total_frames-1} "
                               f"({self.current_frame/self.fps:.1f}s)")
            
            # Update position labels
            if self.current_frame in self.annotations:
                anno = self.annotations[self.current_frame]
                for point in self.keypoint_names:
                    if point in anno and anno[point]:
                        x, y = anno[point]
                        self.pos_labels[point].set(f"({x}, {y})")
                    else:
                        self.pos_labels[point].set("Not set")
            else:
                for point in self.keypoint_names:
                    self.pos_labels[point].set("Not set")
            
            # Update stats
            self.update_stats()
            
            # Update progress bar
            self.update_progress_bar()
    
    def draw_annotations(self, frame):
        display = frame.copy()
        
        if self.current_frame in self.annotations:
            anno = self.annotations[self.current_frame]
            
            points = []
            # Draw points
            for point_type in self.keypoint_names:
                if point_type in anno and anno[point_type]:
                    x, y = anno[point_type]
                    color = self.colors[point_type]
                    
                    # Check if this is a prediction
                    if (self.current_frame in self.predictions and 
                        point_type in self.predictions[self.current_frame]):
                        # Draw with different style for predictions
                        cv2.circle(display, (x, y), self.point_radius, color, 2)
                        # Add confidence text if available
                        if self.current_frame in self.prediction_confidence:
                            conf = self.prediction_confidence[self.current_frame].get(point_type, 0)
                            cv2.putText(display, f"{conf:.2f}", (x + 15, y + 15),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                    else:
                        # Regular annotation (filled)
                        cv2.circle(display, (x, y), self.point_radius, color, -1)
                        cv2.circle(display, (x, y), self.point_radius + 2, (255, 255, 255), 2)
                    
                    # Label
                    cv2.putText(display, point_type[0].upper(), (x + 10, y - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                    points.append((x, y))
            
            # Draw skeleton if enabled - connect all points to center
            if self.skeleton_var.get() and 'center' in anno and anno['center']:
                center_pos = anno['center']
                for point_type in self.keypoint_names:
                    if point_type != 'center' and point_type in anno and anno[point_type]:
                        cv2.line(display, center_pos, anno[point_type], (255, 255, 0), 2)
        
        # Add confidence indicator if this frame has predictions
        if self.current_frame in self.prediction_confidence:
            avg_conf = np.mean(list(self.prediction_confidence[self.current_frame].values()))
            color = (0, 255, 0) if avg_conf > self.conf_var.get() else (0, 0, 255)
            cv2.putText(display, f"Pred Conf: {avg_conf:.2f}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        return display
    
    def get_frame_coordinates(self, event_x, event_y):
        """Convert display coordinates to frame coordinates"""
        if not hasattr(self, 'display_width') or not hasattr(self, 'frame_width'):
            return None, None
        
        # Get the actual size of the label widget
        label_width = self.video_label.winfo_width()
        label_height = self.video_label.winfo_height()
        
        # Calculate centering offset if image is centered in label
        x_offset = (label_width - self.display_width) // 2
        y_offset = (label_height - self.display_height) // 2
        
        # Adjust event coordinates for centering
        adjusted_x = event_x - x_offset
        adjusted_y = event_y - y_offset
        
        # Check if click is within image bounds
        if adjusted_x < 0 or adjusted_x >= self.display_width or adjusted_y < 0 or adjusted_y >= self.display_height:
            return None, None
        
        # Calculate scale factors
        scale_x = self.frame_width / self.display_width
        scale_y = self.frame_height / self.display_height
        
        # Convert to frame coordinates
        x = int(adjusted_x * scale_x)
        y = int(adjusted_y * scale_y)
        
        # Ensure coordinates are within frame bounds
        x = max(0, min(x, self.frame_width - 1))
        y = max(0, min(y, self.frame_height - 1))
        
        return x, y
    
    def get_display_coordinates(self, frame_x, frame_y):
        """Convert frame coordinates to display coordinates"""
        if not hasattr(self, 'display_width') or not hasattr(self, 'frame_width'):
            return None, None
        
        # Get the actual size of the label widget
        label_width = self.video_label.winfo_width()
        label_height = self.video_label.winfo_height()
        
        # Calculate centering offset
        x_offset = (label_width - self.display_width) // 2
        y_offset = (label_height - self.display_height) // 2
        
        scale_x = self.display_width / self.frame_width
        scale_y = self.display_height / self.frame_height
        
        # Convert to display coordinates and add offset
        display_x = int(frame_x * scale_x) + x_offset
        display_y = int(frame_y * scale_y) + y_offset
        
        return display_x, display_y
    
    def find_nearest_point(self, x, y):
        """Find the nearest annotated point to the given coordinates"""
        if self.current_frame not in self.annotations:
            return None
        
        anno = self.annotations[self.current_frame]
        min_dist = float('inf')
        nearest_point = None
        
        for point_type in self.keypoint_names:
            if point_type in anno and anno[point_type]:
                px, py = anno[point_type]
                # Convert to display coordinates for distance calculation
                dpx, dpy = self.get_display_coordinates(px, py)
                if dpx is None:
                    continue
                
                dist = ((x - dpx) ** 2 + (y - dpy) ** 2) ** 0.5
                if dist < min_dist and dist < 25:  # 25 pixel threshold for easier grabbing
                    min_dist = dist
                    nearest_point = point_type
        
        return nearest_point
    
    def on_video_click(self, event):
        if not self.cap:
            return
        
        # Get click coordinates
        x, y = self.get_frame_coordinates(event.x, event.y)
        if x is None:
            return
        
        # In click mode, always move the selected point
        if self.click_mode:
            # Ensure current frame has annotation dict
            if self.current_frame not in self.annotations:
                self.annotations[self.current_frame] = {}
            
            # Set the selected point
            selected = self.point_var.get()
            self.annotations[self.current_frame][selected] = (x, y)
            
            # Auto-advance to next point type
            current_idx = self.keypoint_names.index(selected)
            next_idx = (current_idx + 1) % len(self.keypoint_names)
            self.point_var.set(self.keypoint_names[next_idx])
            
            # Remove from predictions if it was a prediction
            if self.current_frame in self.predictions and selected in self.predictions[self.current_frame]:
                del self.predictions[self.current_frame][selected]
                if not self.predictions[self.current_frame]:  # If no more predictions in frame
                    del self.predictions[self.current_frame]
            
            self.update_frame()
            return
        
        # Original drag mode behavior
        # Check if clicking on an existing point
        nearest = self.find_nearest_point(event.x, event.y)
        
        if nearest:
            # Start dragging
            self.dragging = True
            self.drag_point = nearest
            self.point_var.set(nearest)  # Select the point dragging
        else:
            # Only place new point if not near any existing points
            # Check if all points are already placed
            if self.current_frame in self.annotations:
                anno = self.annotations[self.current_frame]
                all_placed = all(point in anno and anno[point] for point in self.keypoint_names)
                if all_placed:
                    # Don't place new points if all are already set
                    return
            
            # Ensure current frame has annotation dict
            if self.current_frame not in self.annotations:
                self.annotations[self.current_frame] = {}
            
            # Set the selected point
            selected = self.point_var.get()
            self.annotations[self.current_frame][selected] = (x, y)
            
            # Auto-advance to next point type only if it's not already placed
            anno = self.annotations[self.current_frame]
            current_idx = self.keypoint_names.index(selected)
            next_idx = (current_idx + 1) % len(self.keypoint_names)
            next_point = self.keypoint_names[next_idx]
            if next_point not in anno or not anno[next_point]:
                self.point_var.set(next_point)
            
            self.update_frame()
    
    def on_drag(self, event):
        if self.dragging and self.drag_point:
            x, y = self.get_frame_coordinates(event.x, event.y)
            if x is None:
                return
            
            if self.current_frame in self.annotations:
                self.annotations[self.current_frame][self.drag_point] = (x, y)
                self.update_frame()
    
    def on_release(self, event):
        self.dragging = False
        self.drag_point = None
    
    def update_progress_bar(self):
        """Update the visual progress bar showing annotated frames"""
        if not self.cap or self.total_frames == 0:
            return
        
        # Clear canvas
        self.progress_canvas.delete("all")
        
        # Get canvas dimensions - force update to get correct width
        self.progress_canvas.update_idletasks()
        canvas_width = self.progress_canvas.winfo_width()
        canvas_height = 20
        
        # Ensure there is a valid width
        if canvas_width <= 1:
            # Schedule a retry after the GUI has properly initialized
            self.root.after(100, self.update_progress_bar)
            return
        
        # For very long videos, use a sampling approach
        if self.total_frames > 1000:
            # Sample every Nth frame to keep visualization manageable
            sample_rate = max(1, self.total_frames // 1000)
            sampled_frames = range(0, self.total_frames, sample_rate)
            frame_width = canvas_width / len(sampled_frames)
            
            # Draw sampled progress blocks
            for i, frame_num in enumerate(sampled_frames):
                x1 = i * frame_width
                x2 = (i + 1) * frame_width
                
                # Check annotation status in the range
                has_annotation = False
                is_complete = False
                
                # Check a range around this sample point
                check_start = frame_num
                check_end = min(frame_num + sample_rate, self.total_frames)
                
                for check_frame in range(check_start, check_end):
                    if check_frame in self.annotations:
                        has_annotation = True
                        anno = self.annotations[check_frame]
                        if all(point in anno and anno[point] for point in self.keypoint_names):
                            is_complete = True
                            break
                
                # Determine color
                if is_complete:
                    color = '#4CAF50'  # Green for complete
                elif has_annotation:
                    color = '#FFC107'  # Yellow for partial
                else:
                    color = '#f0f0f0'  # Light gray for not annotated
                
                # Draw rectangle
                self.progress_canvas.create_rectangle(x1, 0, x2, canvas_height, 
                                                    fill=color, outline='', tags='progress')
            
            # Draw current frame indicator (scale to sampled view)
            current_sample_pos = (self.current_frame // sample_rate) * frame_width + frame_width / 2
            
        else:
            # Original approach for shorter videos
            frame_width = canvas_width / self.total_frames
            
            # Draw progress blocks
            for frame_num in range(self.total_frames):
                x1 = frame_num * frame_width
                x2 = (frame_num + 1) * frame_width
                
                # Determine color
                if frame_num in self.annotations:
                    anno = self.annotations[frame_num]
                    if all(point in anno and anno[point] for point in self.keypoint_names):
                        color = '#4CAF50'  # Green for complete
                    else:
                        color = '#FFC107'  # Yellow for partial
                else:
                    color = '#f0f0f0'  # Light gray for not annotated
                
                # Draw rectangle
                self.progress_canvas.create_rectangle(x1, 0, x2, canvas_height, 
                                                    fill=color, outline='', tags='progress')
            
            # Current frame position for normal videos
            current_sample_pos = self.current_frame * frame_width + frame_width / 2
        
        # Draw current frame indicator with better visibility
        # White background for contrast
        self.progress_canvas.create_line(current_sample_pos - 1, 0, current_sample_pos - 1, canvas_height, 
                                       fill='white', width=3, tags='indicator-bg')
        self.progress_canvas.create_line(current_sample_pos + 1, 0, current_sample_pos + 1, canvas_height, 
                                       fill='white', width=3, tags='indicator-bg')
        # Red indicator in the middle
        self.progress_canvas.create_line(current_sample_pos, 0, current_sample_pos, canvas_height, 
                                       fill='red', width=2, tags='indicator')
        
        # Add frame number text near cursor for long videos
        if self.total_frames > 800:
            # Add text showing current frame
            text_x = min(current_sample_pos + 20, canvas_width - 40)
            if current_sample_pos > canvas_width - 60:
                text_x = current_sample_pos - 20
            
            self.progress_canvas.create_text(text_x, canvas_height // 2, 
                                           text=f"F{self.current_frame}", 
                                           fill='red', 
                                           font=('Arial', 8, 'bold'),
                                           tags='indicator-text')
    
    def toggle_click_mode(self):
        """Toggle between click-to-move and drag mode"""
        self.click_mode = self.click_mode_var.get()
        
    def copy_previous_frame(self):
        """Copy annotations from previous frame to current frame"""
        if self.current_frame > 0 and self.current_frame - 1 in self.annotations:
            self.annotations[self.current_frame] = self.annotations[self.current_frame - 1].copy()
            # Remove from predictions if it was a prediction
            if self.current_frame in self.predictions:
                del self.predictions[self.current_frame]
            if self.current_frame in self.prediction_confidence:
                del self.prediction_confidence[self.current_frame]
            self.update_frame()
        else:
            messagebox.showwarning("Warning", "No previous frame to copy from")
    
    def on_point_select(self):
        """Update selected point when radio button is clicked"""
        self.selected_point = self.point_var.get()
    
    def skip_frames(self, n):
        if self.cap:
            old_frame = self.current_frame
            self.current_frame = max(0, min(self.current_frame + n, self.total_frames - 1))
            
            # Auto-select head when moving to a new frame
            if old_frame != self.current_frame:
                self.point_var.set('head')
            
            self.update_frame()
    
    def on_slider_change(self, value):
        if self.cap and not self.playing:
            old_frame = self.current_frame
            self.current_frame = int(float(value))
            
            # Auto-select head when moving to a new frame
            if old_frame != self.current_frame:
                self.point_var.set('head')
            
            self.update_frame()
    
    def toggle_play(self):
        self.playing = not self.playing
        self.play_button.config(text="Pause" if self.playing else "Play")
        if self.playing:
            self.play_video()
    
    def play_video(self):
        if self.playing and self.cap:
            self.skip_frames(1)
            if self.current_frame < self.total_frames - 1:
                self.root.after(int(1000/self.fps), self.play_video)
            else:
                self.playing = False
                self.play_button.config(text="Play")
    
    def clear_current_frame(self):
        if self.current_frame in self.annotations:
            del self.annotations[self.current_frame]
            self.update_frame()
    
    def mass_clear_dialog(self):
        """Dialog for mass clearing frames"""
        if not self.cap:
            messagebox.showwarning("Warning", "No video loaded")
            return
        
        dialog = tk.Toplevel(self.root)
        dialog.title("Mass Clear Frames")
        dialog.geometry("300x200")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Instructions
        ttk.Label(dialog, text="Clear annotations from frame range:",
                 font=('Arial', 10)).pack(pady=10)
        
        # Frame inputs
        frame_container = ttk.Frame(dialog)
        frame_container.pack(pady=10)
        
        ttk.Label(frame_container, text="From:").grid(row=0, column=0, padx=5)
        from_var = tk.IntVar(value=0)
        from_entry = ttk.Entry(frame_container, textvariable=from_var, width=10)
        from_entry.grid(row=0, column=1, padx=5)
        
        ttk.Label(frame_container, text="To:").grid(row=0, column=2, padx=5)
        to_var = tk.IntVar(value=self.total_frames-1)
        to_entry = ttk.Entry(frame_container, textvariable=to_var, width=10)
        to_entry.grid(row=0, column=3, padx=5)
        
        # Info label
        info_label = ttk.Label(dialog, text="", foreground="red")
        info_label.pack(pady=5)
        
        def validate_and_clear():
            start = from_var.get()
            end = to_var.get()
            
            if start < 0 or end >= self.total_frames or start > end:
                info_label.config(text="Invalid frame range!")
                return
            
            # Count annotations in range
            count = sum(1 for f in range(start, end+1) if f in self.annotations)
            
            if count == 0:
                info_label.config(text="No annotations in this range")
                return
            
            # Confirm
            if messagebox.askyesno("Confirm Clear", 
                                  f"Clear {count} annotations from frames {start} to {end}?",
                                  parent=dialog):
                # Clear the frames
                for frame in range(start, end+1):
                    if frame in self.annotations:
                        del self.annotations[frame]
                    if frame in self.predictions:
                        del self.predictions[frame]
                    if frame in self.prediction_confidence:
                        del self.prediction_confidence[frame]
                
                self.update_frame()
                messagebox.showinfo("Success", f"Cleared {count} annotations", parent=dialog)
                dialog.destroy()
        
        # Buttons
        button_frame = ttk.Frame(dialog)
        button_frame.pack(pady=20)
        
        ttk.Button(button_frame, text="Clear", 
                   command=validate_and_clear).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Cancel", 
                   command=dialog.destroy).pack(side=tk.LEFT, padx=5)
        
        # Focus on from entry
        from_entry.focus()
        from_entry.select_range(0, tk.END)
    
    def jump_to_frame_dialog(self):
        """Dialog for jumping to a specific frame"""
        if not self.cap:
            messagebox.showwarning("Warning", "No video loaded")
            return
        
        dialog = tk.Toplevel(self.root)
        dialog.title("Jump to Frame")
        dialog.geometry("300x150")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Instructions
        ttk.Label(dialog, text=f"Enter frame number (0-{self.total_frames-1}):",
                 font=('Arial', 10)).pack(pady=10)
        
        # Frame input
        frame_var = tk.IntVar(value=self.current_frame)
        frame_entry = ttk.Entry(dialog, textvariable=frame_var, width=15)
        frame_entry.pack(pady=5)
        
        # Time display
        time_label = ttk.Label(dialog, text="")
        time_label.pack(pady=5)
        
        def update_time(*args):
            try:
                frame = frame_var.get()
                if 0 <= frame < self.total_frames:
                    time_sec = frame / self.fps
                    time_label.config(text=f"Time: {time_sec:.2f}s")
                else:
                    time_label.config(text="Invalid frame number", foreground="red")
            except:
                time_label.config(text="")
        
        frame_var.trace('w', update_time)
        update_time()
        
        def jump():
            try:
                frame = frame_var.get()
                if 0 <= frame < self.total_frames:
                    self.current_frame = frame
                    self.update_frame()
                    dialog.destroy()
                else:
                    messagebox.showerror("Error", f"Frame must be between 0 and {self.total_frames-1}", 
                                       parent=dialog)
            except ValueError:
                messagebox.showerror("Error", "Please enter a valid number", parent=dialog)
        
        # Buttons
        button_frame = ttk.Frame(dialog)
        button_frame.pack(pady=20)
        
        ttk.Button(button_frame, text="Jump", 
                   command=jump).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Cancel", 
                   command=dialog.destroy).pack(side=tk.LEFT, padx=5)
        
        # Bind Enter key
        dialog.bind('<Return>', lambda e: jump())
        
        # Focus and select all
        frame_entry.focus()
        frame_entry.select_range(0, tk.END)
    
    def predict_frames_dialog(self):
        """Dialog for choosing how many frames to predict"""
        if not self.yolo_model:
            messagebox.showwarning("Warning", "No model loaded. Train or load a model first.")
            return
        
        if not self.cap:
            messagebox.showwarning("Warning", "No video loaded")
            return
        
        dialog = tk.Toplevel(self.root)
        dialog.title("Predict Frames")
        dialog.geometry("350x200")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Instructions
        max_frames = self.total_frames - self.current_frame - 1
        ttk.Label(dialog, text=f"Current frame: {self.current_frame}",
                 font=('Arial', 10)).pack(pady=5)
        ttk.Label(dialog, text=f"Frames available: {max_frames}",
                 font=('Arial', 10)).pack(pady=5)
        
        # Input frame
        input_frame = ttk.Frame(dialog)
        input_frame.pack(pady=10)
        
        ttk.Label(input_frame, text="Number of frames to predict:").pack(side=tk.LEFT, padx=5)
        
        num_var = tk.IntVar(value=min(200, max_frames))
        num_entry = ttk.Entry(input_frame, textvariable=num_var, width=10)
        num_entry.pack(side=tk.LEFT, padx=5)
        
        # Quick select buttons
        quick_frame = ttk.Frame(dialog)
        quick_frame.pack(pady=5)
        
        for num in [50, 100, 200, 500]:
            if num <= max_frames:
                ttk.Button(quick_frame, text=str(num), width=6,
                          command=lambda n=num: num_var.set(n)).pack(side=tk.LEFT, padx=2)
        
        def predict():
            try:
                num_frames = num_var.get()
                if num_frames <= 0:
                    messagebox.showerror("Error", "Please enter a positive number", parent=dialog)
                    return
                if num_frames > max_frames:
                    messagebox.showerror("Error", f"Cannot predict more than {max_frames} frames", parent=dialog)
                    return
                
                dialog.destroy()
                self.predict_frames(num_frames)
                
            except ValueError:
                messagebox.showerror("Error", "Please enter a valid number", parent=dialog)
        
        # Buttons
        button_frame = ttk.Frame(dialog)
        button_frame.pack(pady=20)
        
        ttk.Button(button_frame, text="Predict", 
                   command=predict).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Cancel", 
                   command=dialog.destroy).pack(side=tk.LEFT, padx=5)
        
        # Bind Enter key
        dialog.bind('<Return>', lambda e: predict())
        
        # Focus and select all
        num_entry.focus()
        num_entry.select_range(0, tk.END)
    
    def interpolate_frames(self):
        if len(self.annotations) < 2:
            messagebox.showwarning("Warning", "Need at least 2 annotated frames to interpolate")
            return
        
        frames = sorted(self.annotations.keys())
        interpolated = 0
        
        for i in range(len(frames) - 1):
            start_frame = frames[i]
            end_frame = frames[i + 1]
            
            if end_frame - start_frame <= 1:
                continue
            
            start_anno = self.annotations[start_frame]
            end_anno = self.annotations[end_frame]
            
            for f in range(start_frame + 1, end_frame):
                t = (f - start_frame) / (end_frame - start_frame)
                
                self.annotations[f] = {}
                for point in self.keypoint_names:
                    if point in start_anno and point in end_anno:
                        if start_anno[point] and end_anno[point]:
                            x1, y1 = start_anno[point]
                            x2, y2 = end_anno[point]
                            x = int(x1 + t * (x2 - x1))
                            y = int(y1 + t * (y2 - y1))
                            self.annotations[f][point] = (x, y)
                
                interpolated += 1
        
        messagebox.showinfo("Success", f"Interpolated {interpolated} frames")
        self.update_frame()
    
    def auto_track_forward(self):
        """PLACEHOLDER: Optical flow tracking from current frame forward"""
        if self.current_frame not in self.annotations:
            messagebox.showwarning("Warning", "Please annotate current frame first")
            return
        
        # This is a placeholder - optical flow tracking would be implemented here
        messagebox.showinfo("Info", "Auto-tracking not implemented yet (PLACEHOLDER)")
    
    def update_stats(self):
        if not self.total_frames:
            self.stats_var.set("No video loaded")
            return
            
        total_annotated = len(self.annotations)
        complete_frames = sum(1 for anno in self.annotations.values() 
                             if all(point in anno and anno[point] 
                                   for point in self.keypoint_names))
        
        self.stats_var.set(f"Annotated: {total_annotated} frames\n"
                          f"Complete: {complete_frames} frames\n"
                          f"Coverage: {total_annotated/self.total_frames*100:.1f}%")
    
    def save_annotations(self):
        if not self.annotations:
            messagebox.showwarning("Warning", "No annotations to save")
            return

        # Generate default filename from video name
        default_filename = "annotations"
        if self.video_filename:
            # Extract just the filename without path and extension
            base_name = os.path.splitext(os.path.basename(self.video_filename))[0]
            default_filename = base_name
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".json",
            filetypes=[("JSON files", "*.json"), ("All files", "*.*")],
            initialfile=default_filename 
        )
        
        if filename:
            data = {
                'annotations': self.annotations,
                'video_info': {
                    'total_frames': self.total_frames,
                    'fps': self.fps
                },
                'timestamp': datetime.now().isoformat()
            }
            
            with open(filename, 'w') as f:
                json.dump(data, f, indent=2)
            
            self._last_saved_annotations = self.annotations.copy()
            
            messagebox.showinfo("Success", f"Saved {len(self.annotations)} annotations")

    def merge_annotations_with_pairs(self, pairs):
        """Helper method to open merge dialog with prepopulated pairs"""
        self.merge_annotations(prepopulated_pairs=pairs)
    
    def load_annotations(self):
        filename = filedialog.askopenfilename(
            filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
        )
        
        if filename:
            with open(filename, 'r') as f:
                data = json.load(f)
            
            # Convert string keys back to integers
            self.annotations = {int(k): v for k, v in data['annotations'].items()}
            self.update_frame()
            messagebox.showinfo("Success", f"Loaded {len(self.annotations)} annotations")
    
    def merge_annotations(self, prepopulated_pairs=None):
        """Merge multiple annotation files and extract frames from corresponding videos"""
        # Create dialog for merge operation
        dialog = tk.Toplevel(self.root)
        dialog.title("Merge Annotations & Extract Frames")
        dialog.geometry("800x600")
        dialog.transient(self.root)
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Instructions
        ttk.Label(dialog, text="Pair annotation files with their video files:", 
                font=('Arial', 12, 'bold')).pack(pady=10)
        
        # File pairs frame
        pairs_frame = ttk.Frame(dialog)
        pairs_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)
        
        # Treeview for file pairs
        columns = ('Annotation', 'Video', 'Status')
        tree = ttk.Treeview(pairs_frame, columns=columns, show='tree headings', height=10)
        tree.heading('#0', text='#')
        tree.heading('Annotation', text='Annotation File')
        tree.heading('Video', text='Video File')
        tree.heading('Status', text='Status')
        
        tree.column('#0', width=40)
        tree.column('Annotation', width=200)
        tree.column('Video', width=200)
        tree.column('Status', width=100)
        
        # Scrollbar
        scrollbar = ttk.Scrollbar(pairs_frame, orient='vertical', command=tree.yview)
        tree.configure(yscrollcommand=scrollbar.set)
        
        tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Store file pairs
        file_pairs = []
        
        # If there are prepopulated pairs, add them
        if prepopulated_pairs:
            for pair in prepopulated_pairs:
                pair_id = len(file_pairs)
                file_pairs.append({
                    'annotation': pair['annotation'],
                    'video': pair['video'],
                    'status': 'Auto-matched'
                })
                
                tree.insert('', 'end', text=str(pair_id + 1),
                        values=(os.path.basename(pair['annotation']), 
                                os.path.basename(pair['video']),
                                'Auto-matched'))
        
        def add_pair():
            # First select annotation file
            anno_file = filedialog.askopenfilename(
                title="Select annotation file",
                filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
            )
            if not anno_file:
                return
                
            # Then select corresponding video file
            video_file = filedialog.askopenfilename(
                title=f"Select video file for {os.path.basename(anno_file)}",
                filetypes=[("Video files", "*.avi *.mp4 *.mov"), ("All files", "*.*")]
            )
            if not video_file:
                return
            
            # Add to list
            pair_id = len(file_pairs)
            file_pairs.append({
                'annotation': anno_file,
                'video': video_file,
                'status': 'Ready'
            })
            
            # Add to tree
            tree.insert('', 'end', text=str(pair_id + 1),
                    values=(os.path.basename(anno_file), 
                            os.path.basename(video_file),
                            'Ready'))
        
        def remove_selected():
            selected = tree.selection()
            for item in selected:
                idx = int(tree.item(item)['text']) - 1
                if 0 <= idx < len(file_pairs):
                    file_pairs.pop(idx)
                tree.delete(item)
            
            # Renumber items
            for i, child in enumerate(tree.get_children()):
                tree.item(child, text=str(i + 1))
        
        def add_batch():
            """Add multiple annotation files and try to auto-match videos"""
            anno_files = filedialog.askopenfilenames(
                title="Select annotation files",
                filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
            )
            if not anno_files:
                return
            
            # Ask for video directory
            video_dir = filedialog.askdirectory(title="Select directory containing video files")
            if not video_dir:
                return
            
            video_dir = Path(video_dir)
            
            # Try to match files
            for anno_file in anno_files:
                anno_base = Path(anno_file).stem  # filename without extension
                
                # Look for matching video
                matched = False
                for ext in ['.avi', '.mp4', '.mov']:
                    video_path = video_dir / f"{anno_base}{ext}"
                    if video_path.exists():
                        # Add pair
                        pair_id = len(file_pairs)
                        file_pairs.append({
                            'annotation': anno_file,
                            'video': str(video_path),
                            'status': 'Ready'
                        })
                        
                        tree.insert('', 'end', text=str(pair_id + 1),
                                values=(os.path.basename(anno_file), 
                                        os.path.basename(video_path),
                                        'Auto-matched'))
                        matched = True
                        break
                
                if not matched:
                    messagebox.showwarning("No match", 
                                        f"No video found for {os.path.basename(anno_file)}")
        
        def auto_pair_folder():
            """Auto-pair files from a single folder"""
            source_folder = filedialog.askdirectory(
                title="Select folder containing paired .json and .avi files"
            )
            if not source_folder:
                return
            
            # Find all json and video files
            json_files = {}
            video_files = {}
            
            for file in os.listdir(source_folder):
                if file.endswith('.json'):
                    base_name = os.path.splitext(file)[0]
                    json_files[base_name] = os.path.join(source_folder, file)
                elif file.endswith(('.avi', '.mp4', '.mov')):
                    base_name = os.path.splitext(file)[0]
                    video_files[base_name] = os.path.join(source_folder, file)
            
            # Find matches
            matched = 0
            for base_name, json_path in json_files.items():
                if base_name in video_files:
                    # Add pair
                    pair_id = len(file_pairs)
                    file_pairs.append({
                        'annotation': json_path,
                        'video': video_files[base_name],
                        'status': 'Auto-matched'
                    })
                    
                    tree.insert('', 'end', text=str(pair_id + 1),
                            values=(os.path.basename(json_path), 
                                    os.path.basename(video_files[base_name]),
                                    'Auto-matched'))
                    matched += 1
            
            if matched > 0:
                messagebox.showinfo("Success", f"Auto-paired {matched} files from folder")
            else:
                messagebox.showwarning("No matches", "No matching pairs found in folder")
        
        # Buttons frame
        button_frame = ttk.Frame(dialog)
        button_frame.pack(fill=tk.X, padx=20, pady=5)
        
        ttk.Button(button_frame, text="Add Pair", 
                command=add_pair).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Add Batch (Auto-match)", 
                command=add_batch).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Auto-Pair from Folder",   # <-- NEW BUTTON
                command=auto_pair_folder).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Remove Selected", 
                command=remove_selected).pack(side=tk.LEFT, padx=5)
        
        # Progress frame
        progress_frame = ttk.LabelFrame(dialog, text="Progress", padding=10)
        progress_frame.pack(fill=tk.X, padx=20, pady=10)
        
        progress_var = tk.DoubleVar()
        progress_bar = ttk.Progressbar(progress_frame, variable=progress_var, maximum=100)
        progress_bar.pack(fill=tk.X)
        
        status_label = ttk.Label(progress_frame, text="Ready to merge")
        status_label.pack(pady=5)
        
        # Output path selection
        output_frame = ttk.Frame(dialog)
        output_frame.pack(fill=tk.X, padx=20, pady=10)
        
        ttk.Label(output_frame, text="Output folder:").pack(side=tk.LEFT, padx=5)
        output_var = tk.StringVar(value="Not selected")
        output_label = ttk.Label(output_frame, textvariable=output_var, relief=tk.SUNKEN, width=40)
        output_label.pack(side=tk.LEFT, padx=5)
        
        output_path = None
        
        def select_output():
            nonlocal output_path
            folder = filedialog.askdirectory(title="Select output folder for training data")
            if folder:
                output_path = folder  # Just use string, not Path
                output_var.set(folder)
        
        ttk.Button(output_frame, text="Browse...", command=select_output).pack(side=tk.LEFT, padx=5)
        
        def perform_merge_and_export():
            if not file_pairs:
                messagebox.showwarning("Warning", "No file pairs added", parent=dialog)
                return
            
            if not output_path:
                messagebox.showwarning("Warning", "Please select an output folder", parent=dialog)
                return
            
            # Create YOLO structure
            images_dir = os.path.join(output_path, "images")
            labels_dir = os.path.join(output_path, "labels")
            os.makedirs(images_dir, exist_ok=True)
            os.makedirs(labels_dir, exist_ok=True)
            
            # Process each pair
            total_exported = 0
            total_pairs = len(file_pairs)
            
            for pair_idx, pair in enumerate(file_pairs):
                status_label.config(text=f"Processing {os.path.basename(pair['annotation'])}...")
                progress_var.set((pair_idx / total_pairs) * 100)
                dialog.update()
                
                try:
                    # Load annotations
                    with open(pair['annotation'], 'r') as f:
                        data = json.load(f)
                    
                    annotations = data.get('annotations', {})
                    
                    # Open video
                    cap = cv2.VideoCapture(pair['video'])
                    if not cap.isOpened():
                        tree.set(tree.get_children()[pair_idx], 'Status', 'Video error')
                        continue
                    
                    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                    
                    # Process each annotated frame
                    frames_exported = 0
                    for frame_str, anno in annotations.items():
                        frame_num = int(frame_str)
                        
                        # Check if all keypoints present
                        if all(point in anno and anno[point] for point in self.keypoint_names):
                            # Read frame
                            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                            ret, frame = cap.read()
                            
                            if ret:
                                # Create unique filename
                                base_name = os.path.splitext(os.path.basename(pair['annotation']))[0]
                                img_filename = f"{base_name}_frame_{frame_num:06d}.jpg"
                                label_filename = f"{base_name}_frame_{frame_num:06d}.txt"
                                
                                # Save image
                                img_path = os.path.join(images_dir, img_filename)
                                cv2.imwrite(img_path, frame)
                                
                                # Save label in YOLO format
                                label_path = os.path.join(labels_dir, label_filename)
                                with open(label_path, 'w') as f:
                                    # Calculate bounding box
                                    xs = [anno[p][0] for p in self.keypoint_names]
                                    ys = [anno[p][1] for p in self.keypoint_names]
                                    x_min, x_max = min(xs), max(xs)
                                    y_min, y_max = min(ys), max(ys)
                                    
                                    # Add padding
                                    padding = 20
                                    x_min = max(0, x_min - padding)
                                    y_min = max(0, y_min - padding)
                                    x_max = min(frame_width, x_max + padding)
                                    y_max = min(frame_height, y_max + padding)
                                    
                                    # Normalize
                                    cx = (x_min + x_max) / 2 / frame_width
                                    cy = (y_min + y_max) / 2 / frame_height
                                    w = (x_max - x_min) / frame_width
                                    h = (y_max - y_min) / frame_height
                                    
                                    # Write YOLO format
                                    f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                                    
                                    # Add keypoints
                                    for point in self.keypoint_names:
                                        x, y = anno[point]
                                        x_norm = x / frame_width
                                        y_norm = y / frame_height
                                        f.write(f" {x_norm:.6f} {y_norm:.6f} 2")
                                    f.write("\n")
                                
                                frames_exported += 1
                                total_exported += 1
                    
                    cap.release()
                    tree.set(tree.get_children()[pair_idx], 'Status', f'✓ {frames_exported} frames')
                    
                except Exception as e:
                    tree.set(tree.get_children()[pair_idx], 'Status', f'Error: {str(e)[:20]}')
            
            # Create data.yaml
            yaml_content = {
                'path': output_path,
                'train': 'images',
                'val': 'images',
                'names': {0: 'rat'},
                'nc': 1,
                'kpt_shape': [7, 3]
            }
            
            yaml_path = os.path.join(output_path, 'data.yaml')
            with open(yaml_path, 'w') as f:
                yaml.dump(yaml_content, f)
            
            progress_var.set(100)
            status_label.config(text=f"Complete! Exported {total_exported} frames")
            
            messagebox.showinfo("Success", 
                              f"Merged and exported {total_exported} frames to:\n{output_path}",
                              parent=dialog)
        
        # Action buttons at bottom
        button_container = ttk.Frame(dialog)
        button_container.pack(pady=20)
        
        # Merge button - big and centered
        merge_button = ttk.Button(button_container, text="Merge & Export Training Data", 
                                 command=perform_merge_and_export, 
                                 style='Accent.TButton',
                                 width=30)
        merge_button.pack(pady=5)
        
        # Close button
        ttk.Button(button_container, text="Close", 
                   command=dialog.destroy,
                   width=15).pack(pady=5)

    def export_for_training(self):
        """Export annotations in YOLO format for training"""
        if not self.annotations:
            messagebox.showwarning("Warning", "No annotations to export")
            return
        
        export_dir = filedialog.askdirectory(title="Select export directory")
        if not export_dir:
            return
        
        # Create YOLO structure
        images_dir = os.path.join(export_dir, "images")
        labels_dir = os.path.join(export_dir, "labels")
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Export frames and labels
        exported = 0
        for frame_num, anno in self.annotations.items():
            if all(point in anno and anno[point] for point in self.keypoint_names):
                # Save frame
                self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                ret, frame = self.cap.read()
                if ret:
                    img_path = os.path.join(images_dir, f"frame_{frame_num:06d}.jpg")
                    cv2.imwrite(img_path, frame)
                    
                    # Save keypoints in normalized format
                    label_path = os.path.join(labels_dir, f"frame_{frame_num:06d}.txt")
                    with open(label_path, 'w') as f:
                        # Format: class x y (normalized)
                        # Use class 0 for rat with 3 keypoints
                        keypoints = []
                        for point in self.keypoint_names:
                            x, y = anno[point]
                            x_norm = x / self.frame_width
                            y_norm = y / self.frame_height
                            keypoints.extend([x_norm, y_norm, 2])  # 2 = visible
                        
                        # YOLO format: class x_center y_center width height kp1_x kp1_y kp1_v ...
                        # For pose, use dummy bbox around keypoints
                        xs = [anno[p][0] for p in self.keypoint_names]
                        ys = [anno[p][1] for p in self.keypoint_names]
                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)
                        
                        cx = (x_min + x_max) / 2 / self.frame_width
                        cy = (y_min + y_max) / 2 / self.frame_height
                        w = (x_max - x_min) / self.frame_width
                        h = (y_max - y_min) / self.frame_height
                        
                        f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        for kp in keypoints:
                            f.write(f" {kp:.6f}")
                        f.write("\n")
                    
                    exported += 1
        
        # Create data.yaml for YOLO
        yaml_content = {
            'path': export_dir,
            'train': 'images',
            'val': 'images',
            'names': {0: 'rat'},
            'nc': 1,
            'kpt_shape': [7, 3]  # 7 keypoints, 3 values each (x, y, visibility)
        }
        
        with open(os.path.join(export_dir, 'data.yaml'), 'w') as f:
            yaml.dump(yaml_content, f)
        
        messagebox.showinfo("Success", f"Exported {exported} frames for training")
        self.update_frame()
    
    def quick_train_dialog(self):
        """Dialog for quick training with epoch selection"""
        if not YOLO_AVAILABLE:
            messagebox.showerror("Error", "ultralytics not installed!\n\nInstall with: pip install ultralytics")
            return
            
        if len(self.annotations) < 50:
            messagebox.showwarning("Warning", "Need at least 50 annotated frames for training")
            return
        
        if self.is_training:
            messagebox.showwarning("Warning", "Training already in progress")
            return
        
        # Create dialog
        dialog = tk.Toplevel(self.root)
        dialog.title("Quick Train Settings")
        dialog.geometry("400x300")
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Title
        ttk.Label(dialog, text="Training Configuration", 
                 font=('Arial', 12, 'bold')).pack(pady=10)
        
        # Current stats
        complete_frames = sum(1 for anno in self.annotations.values() 
                             if all(point in anno and anno[point] 
                                   for point in self.keypoint_names))
        ttk.Label(dialog, text=f"Complete annotations: {complete_frames} frames").pack(pady=5)
        
        # Model name setting
        name_frame = ttk.Frame(dialog)
        name_frame.pack(pady=10)
        
        ttk.Label(name_frame, text="Model name:").grid(row=0, column=0, padx=5)
        
        # Generate default name
        default_name = f"rat_pose_iter_{self.model_iteration + 1:03d}"
        name_var = tk.StringVar(value=default_name)
        name_entry = ttk.Entry(name_frame, textvariable=name_var, width=25)
        name_entry.grid(row=0, column=1, padx=5)
        
        # Epochs setting
        epoch_frame = ttk.Frame(dialog)
        epoch_frame.pack(pady=10)
        
        ttk.Label(epoch_frame, text="Maximum epochs:").grid(row=0, column=0, padx=5)
        epoch_var = tk.IntVar(value=10)  # Default to 10 epochs
        epoch_spinbox = ttk.Spinbox(epoch_frame, from_=1, to=100, textvariable=epoch_var, width=10)
        epoch_spinbox.grid(row=0, column=1, padx=5)
        
        # Early stopping settings
        early_frame = ttk.LabelFrame(dialog, text="Early Stopping", padding=10)
        early_frame.pack(fill=tk.X, padx=20, pady=10)
        
        patience_var = tk.IntVar(value=3)
        ttk.Label(early_frame, text="Stop if no improvement for:").grid(row=0, column=0, sticky=tk.W)
        patience_frame = ttk.Frame(early_frame)
        patience_frame.grid(row=0, column=1, padx=5)
        patience_spinbox = ttk.Spinbox(patience_frame, from_=0, to=20, textvariable=patience_var, width=10)
        patience_spinbox.pack(side=tk.LEFT)
        ttk.Label(patience_frame, text="epochs").pack(side=tk.LEFT, padx=5)
        
        # Info about early stopping
        info_text = "0 = disable early stopping\n3 = stop if no improvement for 3 epochs"
        ttk.Label(early_frame, text=info_text, font=('Arial', 9), foreground='gray').grid(row=1, column=0, columnspan=2, pady=5)
        
        def start_training():
            model_name = name_var.get().strip()
            if not model_name:
                messagebox.showerror("Error", "Please enter a model name", parent=dialog)
                return
                
            # Check if name already exists
            model_path = os.path.join("models_iteration", model_name)
            if os.path.exists(model_path):
                if not messagebox.askyesno("Overwrite?", 
                                         f"Model '{model_name}' already exists. Overwrite?", 
                                         parent=dialog):
                    return
            
            epochs = epoch_var.get()
            patience = patience_var.get()
            dialog.destroy()
            self.quick_train(epochs=epochs, patience=patience, model_name=model_name)
        
        # Start Training button
        go_button = ttk.Button(dialog, text="Start Training", width=20,
                              command=start_training)
        go_button.pack(pady=20)
        
        # Bind Enter key
        dialog.bind('<Return>', lambda e: start_training())
        
        # Focus on name entry and select all
        name_entry.focus()
        name_entry.select_range(0, tk.END)
    
    def quick_train(self, epochs=10, patience=3, model_name=None):
        """Quick training using current annotations"""
        if not YOLO_AVAILABLE:
            messagebox.showerror("Error", "ultralytics not installed!\n\nInstall with: pip install ultralytics")
            return
            
        if len(self.annotations) < 50:
            messagebox.showwarning("Warning", "Need at least 50 annotated frames for training")
            return
        
        if self.is_training:
            messagebox.showwarning("Warning", "Training already in progress")
            return
        
        self.is_training = True
        
        # Create models directory
        models_dir = "models_iteration"
        os.makedirs(models_dir, exist_ok=True)
        
        # Export to temp directory
        temp_dir = tempfile.mkdtemp()
        self.export_dir = temp_dir
        
        # Create dirs with train/val split
        train_images = os.path.join(temp_dir, "train", "images")
        train_labels = os.path.join(temp_dir, "train", "labels")
        val_images = os.path.join(temp_dir, "val", "images")
        val_labels = os.path.join(temp_dir, "val", "labels")
        
        for d in [train_images, train_labels, val_images, val_labels]:
            os.makedirs(d, exist_ok=True)
        
        # Export current annotations
        self.model_status.set("Exporting training data...")
        self.root.update()
        
        # Export frames with 80/20 train/val split
        exported = 0
        frame_list = [(k, v) for k, v in self.annotations.items() 
                      if all(point in v and v[point] for point in self.keypoint_names)]
        
        np.random.shuffle(frame_list)
        split_idx = int(len(frame_list) * 0.8)
        
        for idx, (frame_num, anno) in enumerate(frame_list):
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = self.cap.read()
            if ret:
                # Determine if train or val
                is_train = idx < split_idx
                img_dir = train_images if is_train else val_images
                label_dir = train_labels if is_train else val_labels
                
                img_path = os.path.join(img_dir, f"frame_{frame_num:06d}.jpg")
                cv2.imwrite(img_path, frame)
                
                # Save labels in YOLO pose format
                label_path = os.path.join(label_dir, f"frame_{frame_num:06d}.txt")
                with open(label_path, 'w') as f:
                    # Calculate bounding box
                    xs = [anno[p][0] for p in self.keypoint_names]
                    ys = [anno[p][1] for p in self.keypoint_names]
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)
                    
                    # Add padding to bbox
                    padding = 20
                    x_min = max(0, x_min - padding)
                    y_min = max(0, y_min - padding)
                    x_max = min(self.frame_width, x_max + padding)
                    y_max = min(self.frame_height, y_max + padding)
                    
                    cx = (x_min + x_max) / 2 / self.frame_width
                    cy = (y_min + y_max) / 2 / self.frame_height
                    w = (x_max - x_min) / self.frame_width
                    h = (y_max - y_min) / self.frame_height
                    
                    # Write YOLO format: class x y w h kp1_x kp1_y kp1_v kp2_x kp2_y kp2_v kp3_x kp3_y kp3_v
                    f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                    
                    # Add keypoints (head, center, tail)
                    for point in self.keypoint_names:
                        x, y = anno[point]
                        x_norm = x / self.frame_width
                        y_norm = y / self.frame_height
                        f.write(f" {x_norm:.6f} {y_norm:.6f} 2")  # 2 = visible
                    f.write("\n")
                
                exported += 1
        
        # Create data.yaml for YOLO
        yaml_content = {
            'path': temp_dir,
            'train': 'train/images',
            'val': 'val/images',
            'names': ['rat'],
            'nc': 1,
            # 7 keypoints with 3 dims each (x, y, visibility)
            'kpt_shape': [7, 3]
        }
        
        yaml_path = os.path.join(temp_dir, 'data.yaml')
        with open(yaml_path, 'w') as f:
            yaml.dump(yaml_content, f)
        
        self.model_status.set(f"Training on {exported} frames...")
        self.root.update()
        
        # Use provided name or generate one
        if model_name:
            # Don't increment iteration counter if custom name
            final_model_name = model_name
        else:
            # Increment iteration for auto-generated names
            self.model_iteration += 1
            final_model_name = f"rat_pose_iter_{self.model_iteration:03d}"
        
        try:
            # Train YOLO model in a separate thread
            def train_thread():
                try:
                    # Initialize YOLO with pose model
                    model = YOLO('yolov8n-pose.pt')  # Start with nano pose model
                    
                    # Train the model
                    results = model.train(
                        data=yaml_path,
                        epochs=epochs,  # User-specified epochs
                        imgsz=640,
                        batch=16,
                        name=final_model_name,
                        project=models_dir,
                        exist_ok=True,
                        device='0' if cv2.cuda.getCudaEnabledDeviceCount() > 0 else 'cpu',
                        verbose=True,
                        patience=patience if patience > 0 else 100,  # High number effectively disables it
                        save=True,
                        workers=4,
                        val=True,  # Enable validation
                        plots=True  # Generate plots
                    )
                    
                    # Load the best model
                    best_model_path = os.path.join(models_dir, model_name, 'weights', 'best.pt')
                    trained_model = YOLO(best_model_path)
                    
                    # Get mAP from results if available
                    mAP = 0
                    try:
                        # Try to get mAP from results
                        if hasattr(results, 'results_dict'):
                            mAP = results.results_dict.get('metrics/mAP50-95', 0)
                    except:
                        pass
                    
                    # Save training metadata
                    metadata = {
                        'iteration': self.model_iteration,
                        'model_name': final_model_name,
                        'frames': exported,
                        'train_frames': split_idx,
                        'val_frames': len(frame_list) - split_idx,
                        'timestamp': datetime.now().isoformat(),
                        'model_path': best_model_path,
                        'epochs': epochs,
                        'patience': patience,
                        'annotations': len(self.annotations),
                        'mAP': mAP
                    }
                    
                    metadata_path = os.path.join(models_dir, f"metadata_{final_model_name}.json")
                    with open(metadata_path, 'w') as f:
                        json.dump(metadata, f, indent=2)
                    
                    # Send completion message through queue
                    self.training_queue.put({
                        'type': 'complete',
                        'model': trained_model,
                        'path': best_model_path,
                        'name': final_model_name,
                        'frames': exported,
                        'mAP': mAP,
                        'message': f"Model '{final_model_name}' trained!\n"
                                  f"Best model saved at: {best_model_path}\n"
                                  f"Trained on {exported} frames ({split_idx} train, {len(frame_list)-split_idx} val)"
                    })
                    
                except Exception as e:
                    self.training_queue.put({
                        'type': 'error',
                        'message': f"Training failed: {str(e)}"
                    })
                finally:
                    # Cleanup temp directory
                    shutil.rmtree(temp_dir, ignore_errors=True)
            
            # Start training in background
            training_thread = threading.Thread(target=train_thread)
            training_thread.daemon = True
            training_thread.start()
            
        except Exception as e:
            self.is_training = False
            messagebox.showerror("Error", f"Failed to start training: {str(e)}")
            self.model_status.set("Training failed")
            shutil.rmtree(temp_dir, ignore_errors=True)
    
    def train_from_folder(self):
        """Train model from a folder containing exported training data"""
        if not YOLO_AVAILABLE:
            messagebox.showerror("Error", "ultralytics not installed!\n\nInstall with: pip install ultralytics")
            return
        
        if self.is_training:
            messagebox.showwarning("Warning", "Training already in progress")
            return
        
        # Ask for the training data folder
        data_folder = filedialog.askdirectory(
            title="Select folder containing training data (with images/, labels/, data.yaml)"
        )
        if not data_folder:
            return
        
        # Check for required structure
        if not os.path.exists(os.path.join(data_folder, 'images')):
            messagebox.showerror("Error", "No 'images' folder found in selected directory")
            return
        if not os.path.exists(os.path.join(data_folder, 'labels')):
            messagebox.showerror("Error", "No 'labels' folder found in selected directory")
            return
        if not os.path.exists(os.path.join(data_folder, 'data.yaml')):
            messagebox.showerror("Error", "No 'data.yaml' file found in selected directory")
            return
        
        # Count training samples
        image_files = [f for f in os.listdir(os.path.join(data_folder, 'images')) if f.endswith('.jpg')]
        image_count = len(image_files)
        if image_count == 0:
            messagebox.showerror("Error", "No images found in the images folder")
            return
        
        # Create dialog for training settings
        dialog = tk.Toplevel(self.root)
        dialog.title("Train from Folder")
        dialog.geometry("500x450")  # Made window bigger
        dialog.transient(self.root)
        dialog.grab_set()
        
        # Center the dialog
        dialog.update_idletasks()
        x = (dialog.winfo_screenwidth() // 2) - (dialog.winfo_width() // 2)
        y = (dialog.winfo_screenheight() // 2) - (dialog.winfo_height() // 2)
        dialog.geometry(f"+{x}+{y}")
        
        # Title
        ttk.Label(dialog, text="Training Configuration", 
                font=('Arial', 12, 'bold')).pack(pady=10)
        
        # Info
        info_frame = ttk.Frame(dialog)
        info_frame.pack(pady=10)
        ttk.Label(info_frame, text=f"Training folder: {os.path.basename(data_folder)}").pack()
        ttk.Label(info_frame, text=f"Images found: {image_count}").pack()
        
        # Model name setting
        name_frame = ttk.Frame(dialog)
        name_frame.pack(pady=10)
        
        ttk.Label(name_frame, text="Model name:").grid(row=0, column=0, padx=5)
        default_name = f"merged_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        name_var = tk.StringVar(value=default_name)
        name_entry = ttk.Entry(name_frame, textvariable=name_var, width=30)
        name_entry.grid(row=0, column=1, padx=5)
        
        # Epochs setting
        epoch_frame = ttk.Frame(dialog)
        epoch_frame.pack(pady=10)
        
        ttk.Label(epoch_frame, text="Maximum epochs:").grid(row=0, column=0, padx=5)
        epoch_var = tk.IntVar(value=25)  # More epochs for merged data
        epoch_spinbox = ttk.Spinbox(epoch_frame, from_=1, to=100, textvariable=epoch_var, width=10)
        epoch_spinbox.grid(row=0, column=1, padx=5)
        
        # Early stopping settings
        early_frame = ttk.LabelFrame(dialog, text="Early Stopping", padding=10)
        early_frame.pack(fill=tk.X, padx=20, pady=10)
        
        patience_var = tk.IntVar(value=5)  # Higher patience for merged data
        ttk.Label(early_frame, text="Stop if no improvement for:").grid(row=0, column=0, sticky=tk.W)
        patience_frame = ttk.Frame(early_frame)
        patience_frame.grid(row=0, column=1, padx=5)
        patience_spinbox = ttk.Spinbox(patience_frame, from_=0, to=20, textvariable=patience_var, width=10)
        patience_spinbox.pack(side=tk.LEFT)
        ttk.Label(patience_frame, text="epochs").pack(side=tk.LEFT, padx=5)
        
        # Train/val split
        split_frame = ttk.Frame(dialog)
        split_frame.pack(pady=10)
        ttk.Label(split_frame, text="Validation split:").grid(row=0, column=0, padx=5)
        split_var = tk.DoubleVar(value=0.2)
        split_spinbox = ttk.Spinbox(split_frame, from_=0.1, to=0.5, increment=0.05,
                                textvariable=split_var, width=10, format="%.2f")
        split_spinbox.grid(row=0, column=1, padx=5)
        ttk.Label(split_frame, text="(0.2 = 20% validation)").grid(row=0, column=2, padx=5)
        
        def start_training():
            model_name = name_var.get().strip()
            if not model_name:
                messagebox.showerror("Error", "Please enter a model name", parent=dialog)
                return
            
            epochs = epoch_var.get()
            patience = patience_var.get()
            val_split = split_var.get()
            
            dialog.destroy()
            
            # Start training
            self.is_training = True
            models_dir = "models_iteration"
            os.makedirs(models_dir, exist_ok=True)
            
            self.model_status.set(f"Training from folder: {os.path.basename(data_folder)}")
            self.root.update()
            
            # Train in background thread
            def train_thread():
                try:
                    # First, split the data into train/val if needed
                    self.training_queue.put({
                        'type': 'status',
                        'text': 'Preparing train/validation split...'
                    })
                    
                    # Create train/val folders
                    train_images = os.path.join(data_folder, 'train', 'images')
                    train_labels = os.path.join(data_folder, 'train', 'labels')
                    val_images = os.path.join(data_folder, 'val', 'images')
                    val_labels = os.path.join(data_folder, 'val', 'labels')
                    
                    # Check if already split
                    if not os.path.exists(train_images):
                        # Create directories
                        for d in [train_images, train_labels, val_images, val_labels]:
                            os.makedirs(d, exist_ok=True)
                        
                        # Get all images
                        all_images = [f for f in os.listdir(os.path.join(data_folder, 'images')) 
                                    if f.endswith('.jpg')]
                        np.random.shuffle(all_images)
                        
                        # Split
                        split_idx = int(len(all_images) * (1 - val_split))
                        
                        # Copy files
                        for i, img_name in enumerate(all_images):
                            img_path = os.path.join(data_folder, 'images', img_name)
                            label_name = img_name.replace('.jpg', '.txt')
                            label_path = os.path.join(data_folder, 'labels', label_name)
                            
                            if i < split_idx:
                                # Train
                                shutil.copy2(img_path, os.path.join(train_images, img_name))
                                if os.path.exists(label_path):
                                    shutil.copy2(label_path, os.path.join(train_labels, label_name))
                            else:
                                # Val
                                shutil.copy2(img_path, os.path.join(val_images, img_name))
                                if os.path.exists(label_path):
                                    shutil.copy2(label_path, os.path.join(val_labels, label_name))
                    
                    # Update data.yaml with absolute paths
                    yaml_path = os.path.join(data_folder, 'data.yaml')
                    with open(yaml_path, 'r') as f:
                        yaml_data = yaml.safe_load(f)
                    
                    # Use absolute paths for train/val
                    yaml_data['path'] = data_folder  # Set to the folder containing data.yaml
                    yaml_data['train'] = 'train/images'
                    yaml_data['val'] = 'val/images'
                    
                    with open(yaml_path, 'w') as f:
                        yaml.dump(yaml_data, f)
                    
                    # Also create a backup of original data.yaml
                    shutil.copy2(yaml_path, yaml_path + '.backup')
                    
                    # Initialize YOLO
                    model = YOLO('yolov8n-pose.pt')
                    
                    # Train
                    self.training_queue.put({
                        'type': 'status',
                        'text': f'Training {model_name} on {image_count} images...'
                    })
                    
                    results = model.train(
                        data=yaml_path,  # Use the yaml path directly
                        epochs=epochs,
                        imgsz=640,
                        batch=16,
                        name=model_name,
                        project=models_dir,
                        exist_ok=True,
                        device='0' if cv2.cuda.getCudaEnabledDeviceCount() > 0 else 'cpu',
                        verbose=True,
                        patience=patience if patience > 0 else 100,
                        save=True,
                        workers=4,
                        val=True,
                        plots=True
                    )
                    
                    # Load best model
                    best_model_path = os.path.join(models_dir, model_name, 'weights', 'best.pt')
                    trained_model = YOLO(best_model_path)
                    
                    # Get mAP
                    mAP = 0
                    try:
                        if hasattr(results, 'results_dict'):
                            mAP = results.results_dict.get('metrics/mAP50-95(P)', 0)
                    except:
                        pass
                    
                    self.training_queue.put({
                        'type': 'complete',
                        'model': trained_model,
                        'path': best_model_path,
                        'name': model_name,
                        'frames': image_count,
                        'mAP': mAP,
                        'message': f"Model '{model_name}' trained on {image_count} images!\n"
                                f"Best model saved at: {best_model_path}"
                    })
                    
                except Exception as e:
                    self.training_queue.put({
                        'type': 'error',
                        'message': f"Training failed: {str(e)}"
                    })
            
            # Start training thread
            training_thread = threading.Thread(target=train_thread)
            training_thread.daemon = True
            training_thread.start()
        
        # Button frame at bottom
        button_frame = ttk.Frame(dialog)
        button_frame.pack(side=tk.BOTTOM, fill=tk.X, pady=20)
        
        # Start Training button - large and centered
        start_button = ttk.Button(button_frame, text="Start Training", 
                                command=start_training,
                                style='Accent.TButton',
                                width=25)
        start_button.pack(pady=(0, 10))
        
        # Cancel button
        ttk.Button(button_frame, text="Cancel",
                command=dialog.destroy,
                width=15).pack()

    def load_model(self):
        """Load a pre-trained YOLO model"""
        if not YOLO_AVAILABLE:
            messagebox.showerror("Error", "ultralytics not installed!")
            return
            
        model_path = filedialog.askopenfilename(
            title="Select YOLO model file",
            filetypes=[("Model files", "*.pt *.pth"), ("All files", "*.*")]
        )
        if model_path:
            try:
                self.yolo_model = YOLO(model_path)
                self.model_path = model_path
                self.model_status.set(f"Model: {os.path.basename(model_path)}")
                messagebox.showinfo("Success", "Model loaded successfully!")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to load model: {str(e)}")
    
    def predict_frames(self, num_frames=200):
        """Predict annotations for the next N frames using actual YOLO model"""
        if not self.yolo_model:
            messagebox.showwarning("Warning", "No model loaded. Train or load a model first.")
            return
        
        if not self.cap:
            messagebox.showwarning("Warning", "No video loaded")
            return
        
        start_frame = self.current_frame
        end_frame = min(start_frame + num_frames, self.total_frames)
        
        predicted = 0
        low_conf_frames = []
        
        # Progress dialog
        progress_window = tk.Toplevel(self.root)
        progress_window.title("Predicting Frames")
        progress_window.geometry("300x100")
        
        progress_var = tk.DoubleVar()
        progress_bar = ttk.Progressbar(progress_window, variable=progress_var, maximum=100)
        progress_bar.pack(fill=tk.X, padx=20, pady=20)
        
        status_label = ttk.Label(progress_window, text="Starting prediction...")
        status_label.pack()
        
        progress_window.update()
        
        for frame_idx in range(start_frame, end_frame):
            # Read frame
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = self.cap.read()
            
            if ret:
                # Run YOLO prediction
                results = self.yolo_model(frame, verbose=False)
                
                if results and len(results) > 0:
                    result = results[0]
                    
                    # Check if there are keypoints
                    if hasattr(result, 'keypoints') and result.keypoints is not None:
                        keypoints = result.keypoints
                        
                        if hasattr(keypoints, 'xy') and keypoints.xy.shape[0] > 0:
                            # Get the first detection (highest confidence)
                            kpts = keypoints.xy[0].cpu().numpy()  # Shape: (3, 2) for 3 keypoints
                            confs = keypoints.conf[0].cpu().numpy() if hasattr(keypoints, 'conf') else np.ones(3)
                            
                            # Create annotation
                            self.annotations[frame_idx] = {}
                            self.predictions[frame_idx] = {}
                            self.prediction_confidence[frame_idx] = {}
                            
                            # Map keypoints to head, center, tail
                            point_names = self.keypoint_names
                            avg_conf = 0
                            
                            for i, point_name in enumerate(point_names):
                                if i < len(kpts):
                                    x, y = int(kpts[i][0]), int(kpts[i][1])
                                    conf = float(confs[i]) if i < len(confs) else 0.5
                                    
                                    self.annotations[frame_idx][point_name] = (x, y)
                                    self.predictions[frame_idx][point_name] = (x, y)
                                    self.prediction_confidence[frame_idx][point_name] = conf
                                    avg_conf += conf
                            
                            avg_conf /= len(point_names)
                            
                            # Track low confidence frames
                            if avg_conf < self.conf_var.get():
                                low_conf_frames.append((frame_idx, avg_conf))
                            
                            predicted += 1
            
            # Update progress
            progress = (frame_idx - start_frame + 1) / (end_frame - start_frame) * 100
            progress_var.set(progress)
            status_label.config(text=f"Processing frame {frame_idx}/{end_frame-1}")
            progress_window.update()
        
        progress_window.destroy()
        
        # Report results
        message = f"Predicted {predicted} frames."
        if low_conf_frames:
            message += f"\n{len(low_conf_frames)} frames have low confidence (< {self.conf_var.get():.2f})"
            if self.auto_advance_low_conf:
                # Jump to first low confidence frame
                self.current_frame = low_conf_frames[0][0]
        
        messagebox.showinfo("Prediction Complete", message)
        self.update_frame()
    
    def review_predictions(self):
        """Jump to frames with low confidence predictions"""
        if not self.prediction_confidence:
            messagebox.showinfo("Info", "No predictions to review")
            return
        
        # Find frames with low confidence
        low_conf_frames = []
        threshold = self.conf_var.get()
        
        for frame, confidences in self.prediction_confidence.items():
            avg_conf = np.mean(list(confidences.values()))
            if avg_conf < threshold:
                low_conf_frames.append((frame, avg_conf))
        
        if not low_conf_frames:
            messagebox.showinfo("Info", f"All predictions have confidence > {threshold:.2f}")
            return
        
        # Sort by confidence (lowest first)
        low_conf_frames.sort(key=lambda x: x[1])
        
        # Jump to lowest confidence frame
        self.current_frame = low_conf_frames[0][0]
        self.point_var.set('head')  # Auto-select head
        self.update_frame()
        
        messagebox.showinfo("Review Mode", 
                          f"Found {len(low_conf_frames)} low confidence frames.\n"
                          f"Currently at frame {self.current_frame} (conf: {low_conf_frames[0][1]:.2f})")
    
    def show_model_stats(self):
        """Show model statistics in a new window"""
        if not self.model_path:
            messagebox.showwarning("Warning", "No model loaded")
            return
        
        # Create stats window
        stats_window = tk.Toplevel(self.root)
        stats_window.title("Model Statistics")
        stats_window.geometry("600x400")
        
        # Create text widget with scrollbar
        text_frame = ttk.Frame(stats_window)
        text_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        text_widget = tk.Text(text_frame, wrap=tk.WORD)
        scrollbar = ttk.Scrollbar(text_frame, command=text_widget.yview)
        text_widget.configure(yscrollcommand=scrollbar.set)
        
        text_widget.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Gather model information
        stats_text = f"Model Information\n"
        stats_text += f"{'='*50}\n\n"
        stats_text += f"Model Path: {self.model_path}\n"
        stats_text += f"Model Name: {os.path.basename(self.model_path)}\n\n"
        
        # Try to load training results if available
        model_dir = os.path.dirname(self.model_path)
        results_file = os.path.join(model_dir, '..', 'results.csv')
        
        if os.path.exists(results_file):
            stats_text += "Training Results\n"
            stats_text += f"{'-'*30}\n"
            try:
                import pandas as pd
                df = pd.read_csv(results_file)
                # Show last few epochs
                stats_text += "Last 5 epochs:\n"
                stats_text += df.tail(5).to_string(index=False)
                stats_text += "\n\n"
                
                # Show best metrics
                if 'metrics/mAP50' in df.columns:
                    best_map = df['metrics/mAP50'].max()
                    best_epoch = df.loc[df['metrics/mAP50'].idxmax(), 'epoch']
                    stats_text += f"Best mAP50: {best_map:.4f} (epoch {int(best_epoch)})\n"
            except:
                stats_text += "Could not parse results file\n"
        
        # Add current annotations info
        stats_text += f"\nCurrent Annotations\n"
        stats_text += f"{'-'*30}\n"
        stats_text += f"Total annotated frames: {len(self.annotations)}\n"
        complete_frames = sum(1 for anno in self.annotations.values() 
                             if all(point in anno and anno[point] 
                                   for point in self.keypoint_names))
        stats_text += f"Complete frames: {complete_frames}\n"
        stats_text += f"Partial frames: {len(self.annotations) - complete_frames}\n"
        
        # Add training history if available
        if self.training_history:
            stats_text += f"\nTraining History\n"
            stats_text += f"{'-'*30}\n"
            for i, hist in enumerate(self.training_history):
                stats_text += f"\nIteration {hist['iteration']}:\n"
                stats_text += f"  Frames: {hist['frames']}\n"
                stats_text += f"  mAP: {hist.get('mAP', 'N/A')}\n"
                stats_text += f"  Time: {hist['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}\n"
        
        text_widget.insert('1.0', stats_text)
        text_widget.config(state='disabled')
        
        # Add close button
        ttk.Button(stats_window, text="Close", 
                   command=stats_window.destroy).pack(pady=10)
    
    def show_training_progress(self):
        """Show mAP vs number of annotated frames plot"""
        if not self.training_history:
            messagebox.showinfo("Info", "No training history available. Train at least one model first.")
            return
        
        # Create plot window
        plot_window = tk.Toplevel(self.root)
        plot_window.title("Training Progress - mAP vs Annotations")
        plot_window.geometry("800x600")
        
        # Create matplotlib figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Extract data for plotting
        iterations = []
        frames = []
        maps = []
        
        for hist in self.training_history:
            if 'mAP' in hist and hist['mAP'] > 0:
                iterations.append(hist['iteration'])
                frames.append(hist['frames'])
                maps.append(hist['mAP'])
        
        if not maps:
            ax.text(0.5, 0.5, 'No mAP data available yet', 
                    horizontalalignment='center', verticalalignment='center',
                    transform=ax.transAxes, fontsize=14)
        else:
            # Create the plot
            ax.plot(frames, maps, 'b-o', linewidth=2, markersize=8)
            ax.set_xlabel('Number of Annotated Frames', fontsize=12)
            ax.set_ylabel('mAP (Mean Average Precision)', fontsize=12)
            ax.set_title('Model Performance vs Training Data Size', fontsize=14)
            ax.grid(True, alpha=0.3)
            
            # Add iteration labels
            for i, (x, y, iter_num) in enumerate(zip(frames, maps, iterations)):
                ax.annotate(f'Iter {iter_num}', (x, y), 
                           textcoords="offset points", xytext=(0,10), ha='center')
            
            # Add trend analysis if enough points
            if len(frames) > 2:
                # Calculate improvement rate
                recent_improvement = 0
                if len(maps) > 1:
                    recent_improvement = (maps[-1] - maps[-2]) / (frames[-1] - frames[-2])
                
                # Add text box with insights
                textstr = f'Latest mAP: {maps[-1]:.4f}\n'
                textstr += f'Total Frames: {frames[-1]}\n'
                textstr += f'Recent Improvement: {recent_improvement:.6f} mAP/frame\n'
                
                if recent_improvement < 0.0001 and len(maps) > 3:
                    textstr += '\n⚠️ Performance plateauing\nConsider reviewing annotations'
                
                props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
                ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=10,
                        verticalalignment='top', bbox=props)
        
        # Embed plot in tkinter window
        canvas = FigureCanvasTkAgg(fig, master=plot_window)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        
        # Add toolbar
        toolbar_frame = ttk.Frame(plot_window)
        toolbar_frame.pack(fill=tk.X)
        
        # Add buttons
        button_frame = ttk.Frame(plot_window)
        button_frame.pack(fill=tk.X, pady=10)
        
        ttk.Button(button_frame, text="Save Plot", 
                   command=lambda: self.save_plot(fig)).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Close", 
                   command=plot_window.destroy).pack(side=tk.LEFT, padx=5)
    
    def save_plot(self, fig):
        """Save the current plot"""
        filename = filedialog.asksaveasfilename(
            defaultextension=".png",
            filetypes=[("PNG files", "*.png"), ("PDF files", "*.pdf"), ("All files", "*.*")]
        )
        if filename:
            fig.savefig(filename, dpi=300, bbox_inches='tight')
            messagebox.showinfo("Success", f"Plot saved to {filename}")

    def show_help(self, section_name):
        """Show a specific help section"""
        tips = get_all_tips()
        
        if section_name not in tips:
            return
        
        # Create help window
        help_window = tk.Toplevel(self.root)
        help_window.title(f"Help - {section_name}")
        help_window.geometry("700x600")
        
        # Create text widget with scrollbar
        text_frame = ttk.Frame(help_window)
        text_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        text_widget = tk.Text(text_frame, wrap=tk.WORD, font=('Courier', 10))
        scrollbar = ttk.Scrollbar(text_frame, command=text_widget.yview)
        text_widget.configure(yscrollcommand=scrollbar.set)
        
        text_widget.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Insert content
        text_widget.insert('1.0', tips[section_name])
        text_widget.config(state='disabled')
        
        # Add close button
        ttk.Button(help_window, text="Close", 
                command=help_window.destroy).pack(pady=10)

    def show_all_tips(self):
        """Show all help sections in one window with tabs"""
        tips = get_all_tips()
        
        # Create help window
        help_window = tk.Toplevel(self.root)
        help_window.title("Complete Guide - Rat Pose Annotator")
        help_window.geometry("800x700")
        
        # Create notebook (tabbed interface)
        notebook = ttk.Notebook(help_window)
        notebook.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Add each section as a tab
        for section_name, content in tips.items():
            # Create frame for this tab
            tab_frame = ttk.Frame(notebook)
            notebook.add(tab_frame, text=section_name)
            
            # Create text widget with scrollbar
            text_widget = tk.Text(tab_frame, wrap=tk.WORD, font=('Courier', 10))
            scrollbar = ttk.Scrollbar(tab_frame, command=text_widget.yview)
            text_widget.configure(yscrollcommand=scrollbar.set)
            
            text_widget.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
            scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
            
            # Insert content
            text_widget.insert('1.0', content)
            text_widget.config(state='disabled')
        
        # Add close button
        ttk.Button(help_window, text="Close", 
                command=help_window.destroy).pack(pady=10)


In [ ]:

root = tk.Tk()
app = RatPoseAnnotator(root)
root.mainloop()

invalid command name "4904176832partial"
    while executing
"4904176832partial"
    ("after" script)


invalid command name "4816899392check_training_status"
    while executing
"4816899392check_training_status"
    ("after" script)
